In [1]:
from pathlib import Path
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

In [2]:
SEED = 42


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


set_seed(SEED)


ROOT = next(
    (
        p
        for p in [
            Path.cwd(),
            *Path.cwd().parents,
        ]
        if ((p / "data").is_dir() and (p / "notebooks").is_dir())
    ),
    Path.cwd(),
)

HDA_DIR = ROOT / "data" / "processed" / "hda"

MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)


if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)
print("HDA directory:", HDA_DIR)

Device: mps
HDA directory: /Users/thonph/Desktop/KLTN/data/processed/hda


In [3]:
X_s_train = np.load(
    HDA_DIR / "X_s_train.npy",
    mmap_mode="c",
)

X_s_val = np.load(
    HDA_DIR / "X_s_val.npy",
    mmap_mode="c",
)

X_s_test = np.load(
    HDA_DIR / "X_s_test.npy",
    mmap_mode="c",
)

y_s_train = np.load(
    HDA_DIR / "y_s_train.npy",
)

y_s_val = np.load(HDA_DIR / "y_s_val.npy")

y_s_test = np.load(HDA_DIR / "y_s_test.npy")


print("Train:", X_s_train.shape, y_s_train.shape)
print("Val:  ", X_s_val.shape, y_s_val.shape)
print("Test: ", X_s_test.shape, y_s_test.shape)

print("dtype:", X_s_train.dtype)

Train: (1441582, 203) (1441582,)
Val:   (308911, 203) (308911,)
Test:  (308911, 203) (308911,)
dtype: float32


In [4]:
SOURCE_DIM = X_s_train.shape[1]
LATENT_DIM = 64


assert SOURCE_DIM == 203

assert X_s_val.shape[1] == SOURCE_DIM
assert X_s_test.shape[1] == SOURCE_DIM

assert X_s_train.dtype == np.float32
assert X_s_val.dtype == np.float32
assert X_s_test.dtype == np.float32

assert set(np.unique(y_s_train)).issubset({0, 1})

assert set(np.unique(y_s_val)).issubset({0, 1})

assert set(np.unique(y_s_test)).issubset({0, 1})


print("SOURCE_DIM:", SOURCE_DIM)
print("LATENT_DIM:", LATENT_DIM)

print("Train attack prevalence:", y_s_train.mean())

SOURCE_DIM: 203
LATENT_DIM: 64
Train attack prevalence: 0.048384344421614586


In [5]:
train_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_train)),
    torch.from_numpy(y_s_train.astype(np.float32)),
)

val_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_val)),
    torch.from_numpy(y_s_val.astype(np.float32)),
)

test_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_test)),
    torch.from_numpy(y_s_test.astype(np.float32)),
)

In [6]:
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


print("Train batches:", len(train_loader))

print("Val batches:", len(val_loader))

Train batches: 2816
Val batches: 604


In [7]:
class SourceEncoder(nn.Module):

    def __init__(
        self,
        input_dim: int,
        latent_dim: int = 64,
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.encoder(x)

In [8]:
class BinaryClassifier(nn.Module):

    def __init__(
        self,
        latent_dim: int = 64,
    ):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, z):
        return self.classifier(z).squeeze(1)

In [9]:
source_encoder = SourceEncoder(
    input_dim=SOURCE_DIM,
    latent_dim=LATENT_DIM,
).to(DEVICE)

classifier = BinaryClassifier(
    latent_dim=LATENT_DIM,
).to(DEVICE)


print(source_encoder)

print(classifier)

SourceEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=203, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
  )
)
BinaryClassifier(
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [10]:
x_batch, y_batch = next(iter(train_loader))

x_batch = x_batch.to(
    DEVICE,
    dtype=torch.float32,
)

y_batch = y_batch.to(
    DEVICE,
    dtype=torch.float32,
)

with torch.no_grad():

    z = source_encoder(x_batch)

    logits = classifier(z)


print("Input:", x_batch.shape)

print("Latent:", z.shape)

print("Logits:", logits.shape)

print("Labels:", y_batch.shape)


assert z.shape[1] == LATENT_DIM

assert logits.shape[0] == y_batch.shape[0]

Input: torch.Size([512, 203])
Latent: torch.Size([512, 64])
Logits: torch.Size([512])
Labels: torch.Size([512])


In [11]:
n_negative = int((y_s_train == 0).sum())

n_positive = int((y_s_train == 1).sum())

pos_weight_value = n_negative / n_positive

print("Normal:", n_negative)

print("Attack:", n_positive)

print("pos_weight:", pos_weight_value)

Normal: 1371832
Attack: 69750
pos_weight: 19.66784229390681


In [12]:
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [13]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


optimizer = torch.optim.AdamW(
    list(source_encoder.parameters()) + list(classifier.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [14]:
def train_one_epoch(
    source_encoder,
    classifier,
    loader,
    criterion,
    optimizer,
    device,
):
    source_encoder.train()
    classifier.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in loader:

        x = x.to(
            device,
            dtype=torch.float32,
        )

        y = y.to(
            device,
            dtype=torch.float32,
        )

        optimizer.zero_grad(set_to_none=True)

        z = source_encoder(x)

        logits = classifier(z)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size

        total_samples += batch_size

    return total_loss / total_samples

In [15]:
@torch.no_grad()
def evaluate(
    source_encoder,
    classifier,
    loader,
    criterion,
    device,
    threshold=0.5,
):
    source_encoder.eval()
    classifier.eval()

    total_loss = 0.0
    total_samples = 0

    all_probs = []
    all_labels = []

    for x, y in loader:

        x = x.to(
            device,
            dtype=torch.float32,
        )

        y = y.to(
            device,
            dtype=torch.float32,
        )

        z = source_encoder(x)

        logits = classifier(z)

        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size

        total_samples += batch_size

        all_probs.append(probs.cpu().numpy())

        all_labels.append(y.cpu().numpy())

    y_true = np.concatenate(all_labels)

    y_prob = np.concatenate(all_probs)

    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "loss": (total_loss / total_samples),
        "pr_auc": average_precision_score(
            y_true,
            y_prob,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    }

    return (
        metrics,
        y_true,
        y_prob,
        y_pred,
    )

In [16]:
MAX_EPOCHS = 30
PATIENCE = 5

best_val_pr_auc = -np.inf

best_state = None

epochs_without_improvement = 0

history = []

In [17]:
for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    train_loss = train_one_epoch(
        source_encoder,
        classifier,
        train_loader,
        criterion,
        optimizer,
        DEVICE,
    )

    (
        val_metrics,
        _,
        _,
        _,
    ) = evaluate(
        source_encoder,
        classifier,
        val_loader,
        criterion,
        DEVICE,
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_pr_auc": val_metrics["pr_auc"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_f1": val_metrics["f1"],
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val PR-AUC: {val_metrics['pr_auc']:.4f} | "
        f"Val ROC-AUC: {val_metrics['roc_auc']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["pr_auc"] > best_val_pr_auc:

        best_val_pr_auc = val_metrics["pr_auc"]

        epochs_without_improvement = 0

        best_state = {
            "epoch": epoch,
            "source_encoder": copy.deepcopy(source_encoder.state_dict()),
            "classifier": copy.deepcopy(classifier.state_dict()),
            "optimizer": copy.deepcopy(optimizer.state_dict()),
            "val_metrics": copy.deepcopy(val_metrics),
        }

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print(f"Early stopping at epoch " f"{epoch}")

        break

Epoch 01 | Train Loss: 0.0771 | Val Loss: 0.0572 | Val PR-AUC: 0.9693 | Val ROC-AUC: 0.9984 | Val F1: 0.8722
Epoch 02 | Train Loss: 0.0599 | Val Loss: 0.0572 | Val PR-AUC: 0.9691 | Val ROC-AUC: 0.9984 | Val F1: 0.8731
Epoch 03 | Train Loss: 0.0588 | Val Loss: 0.0584 | Val PR-AUC: 0.9674 | Val ROC-AUC: 0.9985 | Val F1: 0.8708
Epoch 04 | Train Loss: 0.0578 | Val Loss: 0.0574 | Val PR-AUC: 0.9720 | Val ROC-AUC: 0.9985 | Val F1: 0.8690
Epoch 05 | Train Loss: 0.0568 | Val Loss: 0.0547 | Val PR-AUC: 0.9735 | Val ROC-AUC: 0.9986 | Val F1: 0.8734
Epoch 06 | Train Loss: 0.0597 | Val Loss: 0.0544 | Val PR-AUC: 0.9744 | Val ROC-AUC: 0.9987 | Val F1: 0.8720
Epoch 07 | Train Loss: 0.0564 | Val Loss: 0.0544 | Val PR-AUC: 0.9744 | Val ROC-AUC: 0.9987 | Val F1: 0.8730
Epoch 08 | Train Loss: 0.0569 | Val Loss: 0.0563 | Val PR-AUC: 0.9740 | Val ROC-AUC: 0.9986 | Val F1: 0.8655
Epoch 09 | Train Loss: 0.0556 | Val Loss: 0.0547 | Val PR-AUC: 0.9752 | Val ROC-AUC: 0.9987 | Val F1: 0.8733
Epoch 10 | Train Lo

In [18]:
history_df = pd.DataFrame(history)

history_df

,epoch,train_loss,val_loss,val_pr_auc,val_roc_auc,val_f1
0,1,0.077093,0.057230,0.969320,0.998445,0.872199
1,2,0.059936,0.057194,0.969067,0.998396,0.873058
2,3,0.058787,0.058450,0.967360,0.998466,0.870776
3,4,0.057806,0.057353,0.972015,0.998520,0.869029
4,5,0.056790,0.054743,0.973519,0.998609,0.873371
5,6,0.059688,0.054443,0.974399,0.998651,0.871995
6,7,0.056422,0.054389,0.974396,0.998652,0.873040
7,8,0.056915,0.056259,0.973993,0.998650,0.865532
8,9,0.055608,0.054680,0.975188,0.998695,0.873346
9,10,0.055120,0.054224,0.975253,0.998707,0.872912


In [19]:
best_row = history_df.loc[history_df["val_pr_auc"].idxmax()]

best_row

epoch          28.000000
train_loss      0.052567
val_loss        0.051525
val_pr_auc      0.979646
val_roc_auc     0.998919
val_f1          0.872785
Name: 27, dtype: float64

In [20]:
assert best_state is not None


source_encoder.load_state_dict(best_state["source_encoder"])

classifier.load_state_dict(best_state["classifier"])


print("Restored best epoch:", best_state["epoch"])

print("Best validation PR-AUC:", best_state["val_metrics"]["pr_auc"])

Restored best epoch: 28
Best validation PR-AUC: 0.9796461619858862


In [ ]:
(
    _,
    y_val_true,
    y_val_prob,
    _,
) = evaluate(
    source_encoder,
    classifier,
    val_loader,
    criterion,
    DEVICE,
)

precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_val_true,
    y_val_prob,
)

f1_vals = (
    2
    * precision_vals[:-1]
    * recall_vals[:-1]
    / (precision_vals[:-1] + recall_vals[:-1] + 1e-12)
)

best_threshold_idx = np.argmax(f1_vals)

DECISION_THRESHOLD = float(thresholds[best_threshold_idx])

print("Selected threshold:", DECISION_THRESHOLD)

print("Validation F1 at selected threshold:", f1_vals[best_threshold_idx])

Selected threshold: 0.9438012838363647
Validation F1 at selected threshold: 0.9123813291134242


In [22]:
(
    test_metrics,
    y_test_true,
    y_test_prob,
    y_test_pred,
) = evaluate(
    source_encoder,
    classifier,
    test_loader,
    criterion,
    DEVICE,
    threshold=DECISION_THRESHOLD,
)

In [23]:
cm = confusion_matrix(
    y_test_true,
    y_test_pred,
)

print("Confusion Matrix:")

print(cm)

Confusion Matrix:
[[292370   1595]
 [  1215  13731]]


In [24]:
source_test_prevalence = y_test_true.mean()

print("Source test attack prevalence:", source_test_prevalence)

print("Source test PR-AUC:", test_metrics["pr_auc"])

Source test attack prevalence: 0.048382867
Source test PR-AUC: 0.9787953034411052


In [25]:
checkpoint_path = MODEL_DIR / "source_pretrained.pt"

torch.save(
    {
        "seed": SEED,
        "source_dim": SOURCE_DIM,
        "latent_dim": LATENT_DIM,
        "best_epoch": best_state["epoch"],
        "source_encoder_state_dict": source_encoder.state_dict(),
        "classifier_state_dict": classifier.state_dict(),
        "source_test_metrics": test_metrics,
        "source_test_prevalence": float(source_test_prevalence),
        "decision_threshold": DECISION_THRESHOLD,
        "pos_weight": float(pos_weight_value),
        "source_hidden_dims": [256, 128],
        "classifier_hidden_dim": 32,
    },
    checkpoint_path,
)


print("Saved checkpoint:", checkpoint_path)

Saved checkpoint: /Users/thonph/Desktop/KLTN/models/source_pretrained.pt


In [26]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
)

print(checkpoint.keys())

print("source_dim:", checkpoint["source_dim"])

print("latent_dim:", checkpoint["latent_dim"])

print("best_epoch:", checkpoint["best_epoch"])

dict_keys(['seed', 'source_dim', 'latent_dim', 'best_epoch', 'source_encoder_state_dict', 'classifier_state_dict', 'source_test_metrics', 'source_test_prevalence', 'decision_threshold', 'pos_weight', 'source_hidden_dims', 'classifier_hidden_dim'])
source_dim: 203
latent_dim: 64
best_epoch: 28
